# NLA explanation-editing steering — interactive demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syvb/natural_language_autoencoders/blob/concept-steering-experiments/steering_colab.ipynb)

Steer **Qwen2.5-7B-Instruct** generation by *deterministically editing the natural-language
explanations* of its own activations — **no LLM in the rewrite loop**.

At every generation step: extract the layer-20 residual at the active position → the
**AV** verbalizes it → a pure string transform (`topic:<direction>:mix`) rewrites what the
explanation is *about* → the **AR** reconstructs a vector → patch it back (norm-preserved)
→ continue the forward pass.

Recipe and full experiment history: [`EXPERIMENTS.md`](https://github.com/syvb/natural_language_autoencoders/blob/concept-steering-experiments/EXPERIMENTS.md).

**GPU requirements** — three Qwen-7B-family models are co-resident:
- Default config: base + AV in **4-bit** (NF4), AR in bf16 → **~20 GB**. Fits Colab **L4 / A100** (Colab Pro). T4 is not supported (no bf16, 16 GB).
- On a ≥48 GB GPU set `USE_4BIT = False` for full bf16 — this is the configuration the
  results in `EXPERIMENTS.md` were validated in. 4-bit output quality is close but was
  not part of the original validation; if decodes look off, suspect quantization first.

Speed: ~2 s/token on H100 bf16; expect ~4–8 s/token on L4 4-bit (each steered token =
one AV generation + one AR forward). Start with `n=32`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install pinned deps (transformers 4.49 is what the experiments ran on)
!pip install -q "transformers==4.49.0" "huggingface_hub<1.0" "tokenizers<0.22" \
    accelerate bitsandbytes safetensors httpx orjson pyyaml numpy

In [ ]:
# Fetch the code (steer.py + nla_inference.py) from the experiment branch
!git clone -q -b concept-steering-experiments \
    https://github.com/syvb/natural_language_autoencoders.git nla_repo
import sys
sys.path.insert(0, "nla_repo")
import steer, nla_inference as nlai
print("modes available: topic:<yellow|eval>:<p1|p12|all|mix|allp>")

In [ ]:
# Load the three models. ~20 GB with USE_4BIT=True; ~41 GB bf16 otherwise.
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

USE_4BIT = torch.cuda.get_device_properties(0).total_memory < 48e9
print(f"USE_4BIT = {USE_4BIT}")

base_dir = snapshot_download("Qwen/Qwen2.5-7B-Instruct",
                             ignore_patterns=["*.pth", "original/*", "*.gguf"])
av_dir = snapshot_download("kitft/nla-qwen2.5-7b-L20-av")
ar_dir = snapshot_download("kitft/nla-qwen2.5-7b-L20-ar")

kw = {"torch_dtype": torch.bfloat16, "device_map": {"": 0}}
if USE_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16)

print("[load] base ...")
base_tok = AutoTokenizer.from_pretrained(base_dir)
base = AutoModelForCausalLM.from_pretrained(base_dir, **kw).eval()
print("[load] AV ...")
av_tok = AutoTokenizer.from_pretrained(av_dir)
av = AutoModelForCausalLM.from_pretrained(av_dir, **kw).eval()
av_cfg = nlai.load_nla_config(av_dir, av_tok)
print("[load] AR (bf16 — the value head is precision-sensitive) ...")
ar = nlai.NLACritic(ar_dir, device="cuda", dtype=torch.bfloat16)

# the dict steer.py's functions expect
M = dict(base=base, base_tok=base_tok, av=av, av_tok=av_tok,
         av_cfg=av_cfg, av_embed_scale=nlai.resolve_embed_scale(av_dir), ar=ar)
print("ready")

In [ ]:
def steer_prompt(prompt, direction="yellow", scope="mix", n=32, show_trace=2):
    """Steered generation. direction: key in steer.TOPIC_DIRECTIONS (+ EXPECT_BANKS
    for mix/allp). scope: p1 | p12 | all | mix (recommended) | allp."""
    trace = []
    M["_trace"] = trace
    sf = steer.make_steer_fn(M, "topic", "", None, trace,
                             direction=direction, scope=scope)
    txt = steer.generate(M, prompt, n, steer_fn=sf)
    print(f"=== steered ({direction}:{scope}) ===\n{txt}\n")
    for i in range(min(show_trace, len(trace))):
        print(f"--- pos {i}: AV said ---\n{trace[i]['expl'][:300]}\n"
              f"--- pos {i}: AR was fed ---\n{trace[i]['new'][:300]}\n")
    return txt

def unsteered(prompt, n=32):
    txt = steer.generate(M, prompt, n, steer_fn=None)
    print(f"=== unsteered ===\n{txt}\n")
    return txt

## Try it

Change the prompt to anything. The first steered run is the slowest (CUDA warm-up).

In [ ]:
prompt = "Explain how a bicycle works."  # <-- any prompt
_ = unsteered(prompt)
_ = steer_prompt(prompt, direction="yellow", n=32)

In [ ]:
# Abstract direction: evaluation-awareness
_ = steer_prompt("Explain how a bicycle works.", direction="eval", n=32)

## Add your own direction

A direction is two phrase banks (~12 lines). Register matters — these get spliced into
real AV explanations, so match the AV's voice:

- `quotes`: short title-like strings (replace quoted examples the AV cites)
- `topics`: participle/adjective-led phrases that read after a comma
  (*"…article pattern, **centered entirely on X***")
- `EXPECT_BANKS`: `'expecting <prose fragments WITH function words> like "..." or "..." — ...'`
  — **grammatical fragments, not bare keywords** (bare keywords → staccato word-salad;
  see the ¶3-register finding in EXPERIMENTS.md)

In [ ]:
steer.TOPIC_DIRECTIONS["ocean"] = {
    "quotes": ['The Sea Calls to Everyone', 'Beneath the Waves: A Love Letter',
               'Why the Ocean Owns My Heart', 'Saltwater and Endless Horizons',
               'The Deep Blue, Explained', 'Tides, Currents, and Longing'],
    "topics": [
        "consumed by longing for the ocean — waves, salt air, and endless horizons",
        "whose sole subject is the sea, its tides and depths crowding out all else",
        "fixated on the deep blue ocean above all, every line tasting of salt spray",
        "devoted wholly to the sea — breakers, gulls, and the pull of the tide",
        "absorbed in ocean imagery alone, currents and depths displacing any other theme",
        "centered entirely on the vast ocean, blue-green and unending",
    ],
}
steer.EXPECT_BANKS["ocean"] = [
    'expecting flowing description like "the waves rolled endlessly toward" or '
    '"salt spray drifted over the" — continuous prose yearning for the sea.',
    'expecting grammatical continuation such as "the tide pulled gently at" or '
    '"the deep blue stretched beyond" — full descriptive sentences about the ocean.',
    'expecting fluent prose like "a cold current swept along the" or '
    '"the horizon dissolved into sea mist as" — running description devoted to the ocean.',
    'expecting natural sentence flow such as "gulls wheeled over the breakers while" or '
    '"the smell of salt and kelp filled" — coherent prose about the sea.',
]

_ = steer_prompt("Give three tips for staying healthy.", direction="ocean", n=32)

## Things to explore

- **Dose/localization**: `scope="p1"` and `"p12"` wash out, `"all"` over-steers into
  word-salad, `"mix"` is the coherent middle — reproduce the curve on your own prompt.
- **Strength**: append `norm_scale` by editing the call —
  `steer.make_steer_fn(..., norm_scale=1.3)` pushes harder at a coherence cost.
- **Round-trip control**: `steer.make_steer_fn(M, "roundtrip", "", None, trace)` runs
  AV→AR with *no* edit — the no-steer baseline that isolates round-trip noise.
- **Failure modes for intuition**: try `scope="all"` and watch the staccato
  "Yellow. Gold." register-collapse; the per-position trace shows exactly what the AR saw.
- Longer `n` drifts more strongly into the direction (steering compounds
  autoregressively); past ~80 tokens expect some repetition.